In [ ]:
# peak_pattern_atlas.py 

# Periodic Table 
import mendeleev as mv 
from mendeleev.fetch import fetch_table  

#import moseley as mos 
from . import ElementXRF
#from moseley import PeriodicTable

# plotting 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches 
import matplotlib.colors as mcolors 

import holoviews as hv
from holoviews import opts 
from bokeh.models import HoverTool 

import re
import numpy as np

class PeakPatternAtlas():
    '''Create a PeakPatternAtlas instance. '''

    def __init__(self, EOI=None, excitation_energy_keV=25, h_mm=0.001, x_keVs=None, verbose=True, order='alpha'):
        '''Compute an ordered Peak Pattern Atlas for `excitation_energy_keV=25`.'''

        self.table = PeriodicTable(EOI=EOI)

        self.EOI = [element for element in self.table.EOI if not '#'in element]
        
        self.element_xrf_list = [] 
        self.ptrn_dict_list = []

        for i, element in enumerate(self.EOI):     
            if verbose: 
                print(f'Please wait while computing spectral pattern for element {self.EOI[i]} {i+1}/{len(self.EOI)}...', end='\r') 

            element_xrf = ElementXRF(element, excitation_energy_keV=excitation_energy_keV, h_mm=h_mm, x_keVs=x_keVs)
            ptrn_dict = element_xrf.ptrn_dict

            self.element_xrf_list.append(element_xrf)
            self.ptrn_dict_list.append(ptrn_dict)

        if order == 'alpha':
            alphas = [ptrn['alpha_keV'] for ptrn in self.ptrn_dict_list]
            idxs = np.argsort(alphas)
            
        elif order == 'Z': 
            atomic_numbers = [ptrn['atomic_number'] for ptrn in self.ptrn_dict_list] 
            idxs = np.argsort(atomic_numbers) 

        self.ptrn_dict_list = [self.ptrn_dict_list[idx] for idx in idxs]  
        self.element_xrf_list = [self.element_xrf_list[idx] for idx in idxs]  
        self.yticks = [(i, xrf.element) for i, xrf in enumerate(self.element_xrf_list)]
            
        if verbose: 
            print('Ready building Peak Pattern Atlas!                                   ') 

    
    def plot_atlas(self, moseley=False, return_ax=False): 
        '''Plot holoviews peak pattern atlas from `pattern_dict_list`.''' 
    
        # TODO: add escape and sum peaks 
        # TODO: create separate patterns plotting function for other types of plots 
        
        # STEP 1 
        # create peak pattern dataframe and curve segments 
        # that can be used as plotting data 
    
        ptable = self.table
        colors_dict = ptable.element_colors_dict 
        
        min_size, max_size = 3, 10 # markers
        delta_size = max_size - min_size
        
        df_list = []
        segments_left = []
        segments_right = []
        elements = [] 
        Z_list = []
        
        for i, ptrn_dict in enumerate(self.ptrn_dict_list): 
            
            elem = ptrn_dict['elem'] 
            name = ptrn_dict['name']
            color = colors_dict[elem]
            peaks_x, peaks_y = ptrn_dict['peaks_xy'].T
            peak_labels = ptrn_dict['peak_labels']
            peak_labels = [re.sub('\\d', '', pl) for pl in peak_labels] # strip line number e.g. Ka1 -> Ka 
            atomic_number = ptrn_dict['atomic_number']
            n_rows = len(peaks_x) 
            marker_list = ['circle'] * n_rows 
              
            size = (min_size + delta_size * peaks_y / peaks_y.max()).astype(int) 
                                                
            df = pd.DataFrame(dict(Element=[elem] * n_rows,
                                   Z=atomic_number, 
                                   name=[name] * n_rows, 
                                   Line=peak_labels, 
                                   Energy=peaks_x, 
                                   peaks_y=peaks_y, 
                                   color=[color] * n_rows, 
                                   marker=marker_list, 
                                   size=size)) 
            df_list.append(df) 
    
            segments_left.append(float(peaks_x.min()))
            segments_right.append(float(peaks_x.max()))
            elements.append(elem)
            Z_list.append(atomic_number) 
                              
        ptrns_df = pd.concat(df_list) 
    
        # STEP 2
        # create holoviews Scatter and Curves Overlay elements 
    
        tooltips = [
            ('element', '@name'), 
            ('Line', '@Line'),
            ('Energy (keV)', '@Energy')] 
        
        hover = HoverTool(tooltips=tooltips, attachment='vertical')
    
        energy = hv.Dimension('Energy', unit='keV') 
    
        if moseley: 
            y_dim = 'Z'
        else: 
            y_dim = 'Element' 
    
        scatter = hv.Scatter(ptrns_df, kdims=[energy], vdims=[y_dim, 'color', 'size', 'name', 'marker', 'Line']).opts(color='color', line_color='black', marker='marker', size='size', tools=[hover])
        if moseley: 
            y_values = Z_list
        else: 
            y_values = elements 
            
        # create quarto plot or matplotlib axes  
        if not return_ax:     
            grid_style = {'xgrid_line_color': 'gray', 'minor_xgrid_line_color': 'lightgray', 'ygrid_line_color': 'white', 'ygrid_line_alpha': 0} 
            
            curves_list = [hv.Curve(([x0, x1], [y_val, y_val]), kdims=[energy], vdims=y_dim) for x0, x1, y_val in zip(segments_left, segments_right, y_values)]
            curves = hv.Overlay(curves_list).opts(opts.Curve(color='black', line_width=1, gridstyle=grid_style, show_grid=True)) 
        
            atlas = curves * scatter 
            atlas.opts(padding=0.02, frame_width=450, frame_height=500, title='XRF Peak Pattern Atlas')

        else: 
            fig, ax = plt.subplots()
            atlas = ax
            
    
        return atlas 

    

def _fetch_table(): 
    '''Utility function wrapper for mendeleev periodic table data.  
    
    Make dataframe with selected element attributes for regular part of the periodic table.
    The irregular Lanthanides and Actinides series are rare, so we do not plot them. 
    
    Returns: ptable_regular (dataframe) 
    '''
    
    #ptable_df = mv.get_table('elements') # mendeleev 0.5 
    ptable_df = fetch_table('elements')
    ptable = ptable_df[['atomic_number', 'symbol', 'name', 'group_id', 'period']]
    is_regular = ~np.isnan(ptable_df['group_id'].values)

    ptable_regular = ptable[is_regular]
    
    return ptable_regular 


def colorize(ptable_regular): 
    '''Generate regular element colors dictionary. 
    
    Returns: element_colors_dict '''
    
    n_regular = len(ptable_regular) # 90 rows 
    
    # see: https://matplotlib.org/stable/gallery/color/named_colors.html 
    css4_dict = mcolors.CSS4_COLORS 
        
    # hack to select brightest colors and exclude american double grays 
    css4_names = list(css4_dict.keys()) 
    css4_names = sorted(sorted(css4_names, key=lambda c: tuple(mcolors.rgb_to_hsv(mcolors.to_rgb(c))))) # brightest colors 
    css4_names = [name for name in css4_names if 'gray' not in name] # exclude doubles grays 

    # pick brightest 90 
    n_names = len(css4_names) 
    css4_names = css4_names[n_names - n_regular - 1:n_names - 1]

    element_colors_dict = dict(zip(ptable_regular['symbol'], css4_names))

    return element_colors_dict


def _draw_box(ax, element_attrs, edgecolor=None, facecolor=None):
    '''Draw a box for element *element_attrs*  in subplot *ax*. '''
    
    w = 0.9
    h = 0.9
    
    atomic_number, symbol, name, x, y = element_attrs 
    
    if edgecolor is None: 
        edgecolor = [0.8, 0.8, 1.0]
        
    if facecolor is None: 
        facecolor = [0.95, 0.95, 1.0]
    
    rect = mpatches.Rectangle([x - w/2, y - h/2], h, w, edgecolor=edgecolor, facecolor=facecolor)
    ax.add_patch(rect)
    
    ax.annotate(symbol, [x, y], xytext=[0, -4], textcoords='offset points',
                va='center', ha='center', fontsize=17)
    
    ax.annotate(atomic_number, [x, y], xytext=[0, 12], textcoords='offset points', 
                va='center', ha='center', fontsize=10)
    
    ax.annotate(name, [x, y], xytext=[0, -17], textcoords='offset points',
                va='center', ha='center', fontsize=5)


def _sort_ptrn_list(ptrn_dict_list, order='alpha'): 
    '''Utility function to sort ptrn_dict_list in 'alpha' or 'Z' order. 
    
    Returns: sorted_ptrn_list '''

    if order == 'alpha': 
        alpha_list = [ptrn['alpha_keV'] for ptrn in ptrn_dict_list]
        idxs = np.argsort(alpha_list)
    elif order == 'Z': 
        Z_list = [ptrn['atomic_number'] for ptrn in ptrn_dict_list] 
        idxs = np.argsort(Z_list)
        
    sorted_ptrn_list = [ptrn_dict_list[i] for i in idxs]  
        
    return sorted_ptrn_list


class PeriodicTable(): 
    '''Create a periodic table of elements. '''
    
    
    def __init__(self, EOI=None): 

        if EOI is None: 
            self.EOI = ['#H', '#He', '#Li', '#Be', '#B', '#C', 'N', 'O', 'F', 'Ne', 'Na', 'Mg', 'Al', 
                        'Si', 'P', 'S', 'Cl', 'Ar', 'K', 'Ca', '#Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 
                        'Co', 'Ni', 'Cu', 'Zn', '#Ga', '#Ge', 'As', '#Se', 'Br', '#Kr', '#Rb', 'Sr', 
                        '#Y', '#Zr', '#Nb', '#Mo', '#Tc', '#Ru', 'Rh', '#Pd', 'Ag', 'Cd', '#In', 'Sn', 
                        '#Sb', '#Te', 'I', '#Xe', '#Cs', 'Ba', '#La', '#Hf', '#Ta', '#W', '#Re', '#Os', 
                        '#Ir', '#Pt', 'Au', 'Hg', '#Tl', 'Pb', '#Bi', '#Po', '#At', '#Rn', '#Fr', '#Ra', 
                        '#Ac', '#Rf', '#Db', '#Sg', '#Bh', '#Hs', '#Mt', '#Ds', '#Rg', '#Cn', '#Nh', 
                        '#Fl', '#Mc', '#Lv', '#Ts', '#Og']    
        else: 
            self.EOI = EOI 
     
    
        # fetch regular part of periodic table 
        self.ptable_regular = _fetch_table() 
    
        # generate element colors dict 
        self.element_colors_dict = colorize(self.ptable_regular) 


    def plot_table(self, ax=None, figname=None): 
        '''Create periodic table plot with selected elements colorized.'''
    
        if ax is None: 
            fig, ax = plt.subplots(figsize=[14, 8])
        else: 
            fig = ax.get_figure()
            
        ax.invert_yaxis()
    
        for i, element_attrs in enumerate(self.ptable_regular.values): 
    
            atomic_number, symbol, name, x, y = element_attrs 
            facecolor = self.element_colors_dict[symbol] 
            
            if symbol in self.EOI: 
                edgecolor = 'r' 
            else: 
                edgecolor = None
            _draw_box(ax, element_attrs, edgecolor=edgecolor, facecolor=facecolor)
    
        ax.set_xlim(0, 19)
        ax.set_ylim(8, 0)
        ax.axis('off');
        
        if figname is not None: 
    
            fig.savefig(figname)
    
        return ax